# Quickstart: Your first Venice call in 60 seconds

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sabrinaaquino/venice-cookbook/blob/main/notebooks/00-quickstart.ipynb)

_Venice AI Cookbook_

If you only run one notebook in this workshop, run this one. We will install the SDK, wire up your key, send a chat completion, and benchmark three models so you can pick the right one for the rest of the day.

## What you will build

1. Install the OpenAI Python SDK (Venice is OpenAI-compatible, so the same client works).
2. Authenticate with your `VENICE_API_KEY`.
3. Send a chat completion and stream the response.
4. Benchmark three models on the same prompt and rank them by latency.
5. Decide which notebook to open next.

**Cost:** every prompt in this notebook costs less than $0.001 on Venice's default tier. If you have a Pro key the same calls are free.

## 1. Install

In [ ]:
%pip install --quiet openai requests python-dotenv rich

## 2. Set your API key

Get one at [venice.ai/settings/api](https://venice.ai/settings/api). In Colab, add it under the key icon in the left sidebar with the name `VENICE_API_KEY`. Locally, copy `.env.example` to `.env` and fill it in.

In [ ]:
import os, time, json
from textwrap import shorten

# Pick up the API key from Colab secrets, environment, or interactive prompt
try:
    from google.colab import userdata  # type: ignore
    api_key = userdata.get("VENICE_API_KEY")
    os.environ["VENICE_API_KEY"] = api_key
except Exception:
    api_key = os.environ.get("VENICE_API_KEY")

if not api_key:
    from getpass import getpass
    api_key = getpass("Paste your Venice API key: ").strip()
    os.environ["VENICE_API_KEY"] = api_key

from openai import OpenAI
client = OpenAI(api_key=api_key, base_url="https://api.venice.ai/api/v1")
print("Connected to Venice")

## 3. Your first chat completion

Venice exposes a `/chat/completions` endpoint that mirrors the OpenAI spec. The only thing that changes is the base URL. Everything else (messages, roles, tool calling, JSON mode, streaming) is identical.

In [ ]:
resp = client.chat.completions.create(
    model="kimi-k2-6",
    messages=[
        {"role": "system", "content": "You are a sharp, irreverent assistant. Reply in 2 sentences."},
        {"role": "user", "content": "Pitch Venice to a developer in one tweet."},
    ],
)
print(resp.choices[0].message.content)

## 4. Stream tokens as they arrive

In [ ]:
stream = client.chat.completions.create(
    model="kimi-k2-6",
    messages=[{"role": "user", "content": "Write a 4-line haiku about end-to-end encryption."}],
    stream=True,
)
for chunk in stream:
    delta = chunk.choices[0].delta.content or ""
    print(delta, end="", flush=True)
print()

## 5. Benchmark three models

Same prompt, three flagship models side by side. We report wall-clock latency, tokens used, and the first 120 characters of each response so you can eyeball quality. Pandas makes the table render nicely both in Colab and on GitHub.

In [ ]:
import pandas as pd

PROMPT = "Explain why a Trusted Execution Environment is different from a regular cloud VM, in 3 bullets."
MODELS = ["kimi-k2-6", "claude-opus-4-6", "zai-org-glm-5-1"]

rows = []
for model in MODELS:
    t0 = time.perf_counter()
    r = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": PROMPT}],
    )
    elapsed = time.perf_counter() - t0
    rows.append({
        "model": model,
        "latency_s": round(elapsed, 2),
        "prompt_tokens": r.usage.prompt_tokens,
        "completion_tokens": r.usage.completion_tokens,
        "preview": shorten(r.choices[0].message.content.replace("\n", " "), 120),
    })

df = pd.DataFrame(rows).sort_values("latency_s").reset_index(drop=True)
df

## 6. Inspect what Venice actually sees

Venice's default `Private` mode keeps zero logs of prompts or completions. To prove the API is plain HTTPS underneath, here is the raw `requests` version of the same call. Useful for debugging and for porting to languages without an OpenAI SDK.

In [ ]:
import requests

r = requests.post(
    "https://api.venice.ai/api/v1/chat/completions",
    headers={"Authorization": f"Bearer {api_key}", "Content-Type": "application/json"},
    json={
        "model": "kimi-k2-6",
        "messages": [{"role": "user", "content": "Say hi in one word."}],
    },
    timeout=30,
)
print("status:", r.status_code)
print("model:", r.json()["model"])
print("text:", r.json()["choices"][0]["message"]["content"])

## 7. Where to go next

| If you want to... | Open |
|---|---|
| Build a chatbot or agent | `01-chat-completions.ipynb` |
| Build search, RAG, or recommendations | `02-embeddings-and-rag.ipynb` |
| Generate images | `03-image-generation.ipynb` |
| Talk to or transcribe users | `04-audio-tts-stt.ipynb` |
| Generate video | `05-video-generation.ipynb` |
| Use named personas | `06-characters.ipynb` |
| Pay with a wallet (no API key) | `07-x402-wallet-payments.ipynb` |
| Make Venice unable to read your prompt | `08-e2ee-encryption.ipynb` |
| Web search, scrape, parse PDFs, talk to any blockchain | `09-tools-and-rpc.ipynb` |

Have fun. Break things. Ping the [Venice Discord](https://discord.gg/venice-ai) if you get stuck.